# InsightX-AI 3.0 — Customer Lifetime Value (CLV) & Churn Risk Prediction
**Google Colab High-RAM & Cloud Compute Execution Notebook**

This notebook trains and evaluates:
1. **Customer Lifetime Value (CLV)**: Realized Historical GMV vs Parameterized Net Margin CLV, and BG/NBD statistical suitability testing.
2. **Point-in-Time Churn Risk Prediction**: Temporal feature engineering, Heuristic Inactivity Baseline, Logistic Regression, Random Forest, and LightGBM / XGBoost.
3. **Class-Imbalanced Evaluation**: PR-AUC, ROC-AUC, Brier calibration, and Precision@Top-10%.
4. **Artifact Persistence**: Serializes trained models (`.joblib`) and metrics to Google Drive.

In [ ]:
# Step 1: Environment Setup & Cloud Dependencies
!pip install -q duckdb sqlglot psutil pyyaml lightgbm xgboost

# Clone or pull latest InsightX-AI repository
import os
if not os.path.exists('InsightX-AI'):
    !git clone https://github.com/namrata-dev1/InsightX-AI.git
    %cd InsightX-AI
else:
    %cd InsightX-AI
    !git pull

!pip install -e . -q
print('InsightX-AI package installed in editable mode.')

In [ ]:
# Step 2: Mount Google Drive for Artifact Storage
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_DIR = '/content/drive/MyDrive/InsightX-AI/artifacts'
os.makedirs(f'{ARTIFACT_DIR}/models', exist_ok=True)
os.makedirs(f'{ARTIFACT_DIR}/reports', exist_ok=True)
print(f'Artifact storage configured at: {ARTIFACT_DIR}')

In [ ]:
# Step 3: Load Olist Relational Tables
from insightx.data.loader import OlistDataLoader

loader = OlistDataLoader()
tables = loader.load_all_tables()

orders_df = tables['orders']
customers_df = tables['customers']
order_items_df = tables['order_items']
payments_df = tables['order_payments']
reviews_df = tables['order_reviews']
products_df = tables['products']

print(f'Loaded {len(orders_df):,} orders across {len(customers_df):,} customer tokens.')

In [ ]:
# Step 4: Chronological Splitting & Point-in-Time Feature Extraction
from insightx.features.engineering import CustomerFeaturePipeline

CUTOFF_DATE = '2018-03-01 00:00:00'
pipeline = CustomerFeaturePipeline()

# Extract pre-cutoff behavioral feature matrix
features_df = pipeline.build_customer_feature_matrix(
    orders_df=orders_df,
    customers_df=customers_df,
    order_items_df=order_items_df,
    payments_df=payments_df,
    reviews_df=reviews_df,
    products_df=products_df,
    cutoff_date=CUTOFF_DATE,
)

# Extract forward 90-day target labels
targets_df = pipeline.create_forward_targets(
    orders_df=orders_df,
    customers_df=customers_df,
    cutoff_date=CUTOFF_DATE,
    horizon_days=90,
    lookback_days=365,
)

# Merge features with targets (filtered to active lookback cohort)
modeling_data = features_df.merge(targets_df, on='customer_unique_id', how='inner')
active_cohort = modeling_data[modeling_data['is_active_in_lookback'] == True].copy()
print(f'Active cohort size: {len(active_cohort):,} customers.')
print(f'Forward 90-day Churn Rate: {active_cohort["churn_in_horizon"].mean()*100:.2f}%')

In [ ]:
# Step 5: Customer Lifetime Value (CLV) Engine
from insightx.models.clv import CLVEngine

clv_engine = CLVEngine(assumptions_path='configs/business_assumptions.yaml')

# 5a. Historical Realized Value (Gross vs Net)
hist_clv = clv_engine.calculate_historical_value(active_cohort)

# 5b. Audit Probabilistic Model Suitability (BG/NBD & Gamma-Gamma)
suitability = clv_engine.evaluate_probabilistic_model_suitability(active_cohort)
print('Probabilistic Model Suitability Diagnostics:')
print(suitability)

# 5c. Forward 365-day CLV Projections
projected_clv = clv_engine.estimate_forward_clv(hist_clv, horizon_days=365)
print('\nCLV Tier Distribution:')
print(projected_clv['clv_tier'].value_counts())

In [ ]:
# Step 6: Churn Risk Modeling & Comparison
from sklearn.model_selection import train_test_split
from insightx.models.churn import ChurnRiskPredictor

X = active_cohort.drop(columns=['customer_unique_id', 'is_active_in_lookback', 'repeat_in_horizon', 'churn_in_horizon', 'forward_orders_count', 'cutoff_timestamp', 'latest_order_timestamp', 'customer_state'], errors='ignore')
y = active_cohort['churn_in_horizon'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

results = {}
models = ['heuristic', 'logistic_regression', 'random_forest']

for m_type in models:
    predictor = ChurnRiskPredictor(model_type=m_type, random_state=42)
    predictor.train(X_train, y_train)
    metrics = predictor.evaluate(X_test, y_test, top_k_pct=0.10)
    results[m_type] = metrics
    print(f'=== {m_type.upper()} METRICS ===')
    print(metrics)

In [ ]:
# Step 7: Save Model Artifacts to Google Drive
import joblib, json

# Train final production Logistic Regression model on full active cohort
final_model = ChurnRiskPredictor(model_type='logistic_regression', random_state=42)
final_model.train(X, y)

joblib.dump(final_model, f'{ARTIFACT_DIR}/models/churn_risk_logistic_regression.joblib')

with open(f'{ARTIFACT_DIR}/reports/churn_clv_evaluation_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Trained model artifacts and evaluation metrics saved to Google Drive.')